# Colab training smoke (Qucs + multiturn GRPO)

**Prerequisite:** Qucs smoke already works (`SMOKE OK`).

**Goal:** Prove Colab can run the smallest useful training path with **real Qucs rewards**.

1. Runtime: **GPU** (T4 is enough for a micro-run)
2. Phase A: `multiturn_train --dry-run` (real Qucs, no model)
3. Phase B: 1-step micro train (`tasks=1`, `generations=2`, short turns)
4. Success:
   - dry-run prints JSON with finite `reward`
   - micro-run writes `train.log` containing `step_stats=` and saves `final_lora`

Uses [Qucs-S 26.1.1 AppImage](https://github.com/ra3xdh/qucs_s/releases/tag/26.1.1) + repo `training/multiturn_train.py`.


## 0) GPU check


In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — Phase B will fail; Phase A dry-run can still run")


## 1) Clone / update repo


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/isaacguo/qucs-llm-optimizer.git"
REPO_DIR = Path("/content/qucs-llm-optimizer")

if not (REPO_DIR / "src" / "qucs_sim.py").exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only || true

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())


## 2) Install Xvfb (AppImage Qt = xcb only)


In [ ]:
!apt-get -qq update
!apt-get -qq install -y xvfb libxkbcommon-x11-0 libxcb-xinerama0 libxcb-cursor0
!which Xvfb


## 3) Download + extract Qucs-S AppImage


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
QUCS_DIR = Path("/content/qucs-s-appimage")
QUCS_DIR.mkdir(parents=True, exist_ok=True)
!python scripts/setup_qucs_appimage.py --dir {QUCS_DIR}


## 4) Wire Qucs env into this runtime


In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "src"))

from scripts.qucs_appimage import apply_qucs_env, find_qucs_binaries

extract = Path("/content/qucs-s-appimage/squashfs-root")
binaries = find_qucs_binaries(extract)
env = apply_qucs_env(binaries, extract_root=extract)
for k in (
    "QUCS_S",
    "QUCSATOR_RF",
    "DISPLAY",
    "QT_QPA_PLATFORM",
    "QT_PLUGIN_PATH",
    "LD_LIBRARY_PATH",
):
    print(f"{k}={os.environ.get(k)}")

from qucs_sim import resolve_qucs_s, resolve_qucsator
print("resolve_qucs_s:", resolve_qucs_s())
print("resolve_qucsator:", resolve_qucsator())


## 5) Install uv + Python 3.12 train deps

`pyproject.toml` requires Python `>=3.12,<3.13`. Colab system Python may differ — we pin via uv.


In [ ]:
from pathlib import Path
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = str(Path.home() / ".local" / "bin") + os.pathsep + os.environ.get("PATH", "")
!which uv && uv --version
!uv python install 3.12
!uv sync --python 3.12 --extra train


## 6) Phase A — multiturn dry-run (real Qucs, no model)


In [ ]:
import os
from pathlib import Path

# Ensure uv is on PATH for this shell cell too
os.environ["PATH"] = str(Path.home() / ".local" / "bin") + os.pathsep + os.environ.get("PATH", "")

!uv run --python 3.12 python -m training.multiturn_train --dry-run --start-seed 9001 --max-turns 3 --patience 2 --output-dir outputs/colab-train-smoke


## 7) Phase B — GPU micro train (1 step)

Expect ~minutes on T4. Watch for `step_stats=` and `saved final LoRA`.


In [ ]:
import os
from pathlib import Path

os.environ["PATH"] = str(Path.home() / ".local" / "bin") + os.pathsep + os.environ.get("PATH", "")

!uv run --python 3.12 python -m training.multiturn_train \
  --steps 1 \
  --tasks-per-step 1 \
  --generations 2 \
  --max-turns 3 \
  --patience 2 \
  --save-every 1 \
  --start-seed 9001 \
  --output-dir outputs/colab-train-smoke \
  --max-seq-length 1024


## 8) Verify artifacts


In [ ]:
from pathlib import Path

out = Path("outputs/colab-train-smoke")
train_log = out / "train.log"
completions = out / "completions.jsonl"
final_lora = out / "final_lora"

print("train.log exists:", train_log.exists())
print("completions.jsonl exists:", completions.exists())
print("final_lora exists:", final_lora.exists())

if train_log.exists():
    text = train_log.read_text()
    print("has step_stats:", "step_stats=" in text)
    print("--- last 30 log lines ---")
    print("\n".join(text.splitlines()[-30:]))

assert train_log.exists(), "missing train.log"
assert "step_stats=" in train_log.read_text(), "missing step_stats in train.log"
assert final_lora.exists(), "missing final_lora — Phase B did not finish"
print("TRAIN SMOKE OK")


## If this fails

| Symptom | Likely cause |
|---------|--------------|
| dry-run Qt/display error | Re-run cells 2–4; need `DISPLAY=:99` and `QT_QPA_PLATFORM=xcb` |
| `uv sync` Python version error | Confirm `uv python install 3.12` succeeded |
| CUDA / transformers import error | Runtime → change to GPU; restart session |
| OOM during Phase B | Lower `--generations` to 2 (already), or `--max-seq-length 768` |
| Missing `step_stats` | Old code — `git pull` and re-run |
